## Create a compute instance with custom conda environment

Note:
If you see the following cell has error:
```
AzureCliCredential: Please run 'az login' to set up an account and login to the tenant
```

relogin from powershell
```powershell
az logout
az account clear
az login --tenant 00000000-0000-0000-0000-000000000000
```

Reference:
* https://learn.microsoft.com/en-us/azure/machine-learning/how-to-manage-compute-instance?view=azureml-api-2&tabs=python
* create aml compute instance basic https://learn.microsoft.com/en-us/azure/machine-learning/how-to-create-compute-instance?view=azureml-api-2&tabs=python

In [1]:
import azure.ai.ml as aml
print(f"Azure ML SDK version: {aml.__version__}")

Azure ML SDK version: 1.32.0


In [2]:
# get a handle to the workspace
from azure.ai.ml import MLClient
from dotenv import load_dotenv
from azure.ai.ml.entities import ComputeInstance, AmlCompute
from azure.ai.ml.entities import ComputeInstanceSshSettings
from azure.core.exceptions import ResourceExistsError

import datetime
import os
config_file_name = "swedencentral.env"
config_file_path = os.path.join(".", "config", config_file_name)
load_dotenv(dotenv_path=config_file_path, override=True)

from utils.amlauth import AuthHelper
settings = AuthHelper.load_settings()
credential = AuthHelper.test_credential()

ml_client = MLClient(
    credential, settings.subscription_id, settings.resource_group, settings.workspace
)

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


In [3]:
# get the location of the current workspace
workspace_location = ml_client.workspaces.get(settings.workspace).location
print(f"Workspace location: {workspace_location}")

Workspace location: swedencentral


In [4]:
# load the public SSH key content from the specified file path
ssh_public_key_path = os.path.expanduser(os.path.join("~", ".ssh", settings.ssh_pub_key_name))
# print(f"Loading SSH public key from: {ssh_public_key_path}")
with open(ssh_public_key_path, "r") as key_file:
    ssh_public_key_content = key_file.read().strip()
print(f"SSH public key content loaded successfully.")

SSH public key content loaded successfully.


#### Reference
https://learn.microsoft.com/en-us/azure/machine-learning/how-to-customize-compute-instance?view=azureml-api-2


Use 
```
curl -s https://api.ipify.org
```
to get ip and create an NSP in azure to allow local access.

Storage account network, change from disable to "Secured by perimeter"

In [ ]:
# Upload setup files to the AML default blob store as a persistent reference copy.
#
# WHY blob and not file share:
#   Azure Files REST data-plane with OAuth (token_intent="backup") is a Microsoft-internal
#   feature reserved for Azure Backup service. Even with "Storage File Data Privileged
#   Contributor" RBAC, the service rejects OAuth tokens from regular callers (403).
#   From a local machine you cannot reach the file share's private endpoint (VNet-only),
#   and SMB (port 445) is blocked by the Network Security Perimeter.
#
# Blob REST API DOES work: OAuth is fully supported over HTTPS (port 443).
# Requires: "Storage Blob Data Contributor" on the storage account.
#
# NOTE: Cell 10 (CI creation) uses the inline ARM approach — these blobs are a
#       reference copy only and are NOT required for CI creation.

# from azure.storage.blob import BlobServiceClient

# def get_blob_service_client(ml_client, credential):
#     """Build a BlobServiceClient for the AML workspace storage account."""
#     arm_id = ml_client.workspaces.get(ml_client.workspace_name).storage_account
#     storage_account_name = arm_id.split("/")[-1]
#     account_url = f"https://{storage_account_name}.blob.core.windows.net"
#     print(f"Storage account : {storage_account_name}")
#     return BlobServiceClient(account_url=account_url, credential=credential)

# def get_default_blob_container(ml_client):
#     """Resolve the default AML blob container name via the workspaceblobstore datastore."""
#     return ml_client.datastores.get("workspaceblobstore").container_name

# def upload_file_to_blob(blob_svc, container_name, local_file_path, blob_path, force_overwrite=False):
#     """Upload a local file to the AML default blob container."""
#     blob_client = blob_svc.get_blob_client(container=container_name, blob=blob_path)
#     if not force_overwrite and blob_client.exists():
#         print(f"⏭️  Exists (skipped) : {blob_path}")
#         return
#     with open(local_file_path, "rb") as f:
#         blob_client.upload_blob(f, overwrite=True)
#     print(f"✅ Uploaded : '{local_file_path}' → blob:{blob_path}")

# def list_blob_prefix(blob_svc, container_name, prefix):
#     """List blobs under a prefix to verify uploads."""
#     for b in blob_svc.get_container_client(container_name).list_blobs(name_starts_with=prefix):
#         print(f"  📄 {b.name}")

# # ── Build blob client ─────────────────────────────────────────────────────────
# blob_svc = get_blob_service_client(ml_client, credential)
# container_name = get_default_blob_container(ml_client)
# print(f"Container       : {container_name}")

# # ── Blob paths (reference copies — CI creation uses inline script, not these) ─
# blob_setup_script = "setup-scripts/2_setup_conda_env.sh"
# blob_conda_env    = "setup-scripts/slm_conda_sft.yaml"

# # ── Upload (set force_overwrite=True to replace existing files) ───────────────
# upload_file_to_blob(blob_svc, container_name, "2_setup_conda_env.sh", blob_setup_script)
# upload_file_to_blob(blob_svc, container_name, os.path.join("environments", "slm_conda_sft.yaml"), blob_conda_env)

# # ── Verify ────────────────────────────────────────────────────────────────────
# print("\nBlob store contents under setup-scripts/:")
# list_blob_prefix(blob_svc, container_name, "setup-scripts/")


Storage account : amlworkspaceyw3405831058
Container       : azureml-blobstore-bae0a300-6d85-4254-8b17-790d9c6d7a5e
✅ Uploaded : '2_setup_conda_env.sh' → blob:setup-scripts/2_setup_conda_env.sh
✅ Uploaded : 'environments/slm_conda_sft.yaml' → blob:setup-scripts/slm_conda_sft.yaml

Blob store contents under setup-scripts/:
  📄 setup-scripts/2_setup_conda_env.sh
  📄 setup-scripts/slm_conda_sft.yaml


In [30]:
# Create a ComputeInstance with an inline creation script.
#
# WHY not pure SDK (ComputeInstance + ScriptReference)?
#   ScriptReference(path=...) only accepts a Notebooks file-share path, which is
#   unreachable from a local machine behind a Network Security Perimeter.
#   creationScript runs ONCE at provisioning — it cannot be patched onto an
#   existing CI after the fact. startupScript (runs every boot) can be patched
#   but is not suitable for a one-time conda env install.
#
# APPROACH — SDK-first hybrid:
#   1. Build the full ComputeInstance with the Python SDK (all normal params).
#   2. Serialize it to an ARM REST body via _to_rest_object().serialize().
#   3. Inject the inline creationScript into the serialized body.
#   4. PUT the body via requests — same endpoint the SDK would use.
#   This keeps all CI params readable in Python SDK style while adding the
#   inline script that the SDK entity doesn't expose directly.
#
# HOW combined_script works:
#   - Reads slm_conda_sft.yaml from disk and writes it as a bash heredoc to
#     /tmp/aml-setup/slm_conda_sft.yaml on the CI at provisioning time.
#   - Then runs 2_setup_conda_env.sh which references /tmp/aml-setup/slm_conda_sft.yaml.
#   - ENV_NAME is passed explicitly to the sudo heredoc (sudo -i starts a clean
#     login shell that does NOT inherit environment variables).
import base64
import re
import requests
import time as _time
from azure.ai.ml.entities import SetupScripts, ScriptReference

# ── 1. Read slm_conda_sft.yaml and the setup shell script from disk ───────────
with open(os.path.join("environments", "slm_conda_sft.yaml"), "r") as f:
    yaml_content = f.read()

with open("2_setup_conda_env.sh", "r") as f:
    setup_script = f.read()

# Strip the shebang + set -e header so we can prepend them once in the combined script.
# The setup script on disk already has the correct yaml path (/tmp/aml-setup/...).
setup_body = re.sub(r"^#!/bin/bash\n\s*set -e\n", "", setup_script).lstrip("\n")

# ── 2. Build one self-contained bash script ───────────────────────────────────
# Structure of combined_script on the CI:
#
#   #!/bin/bash
#   set -e
#
#   mkdir -p /tmp/aml-setup
#   cat > /tmp/aml-setup/slm_conda_sft.yaml << 'YAML_HEREDOC'
#   <contents of environments/slm_conda_sft.yaml>          ← embedded at notebook run time
#   YAML_HEREDOC
#
#   <contents of 2_setup_conda_env.sh without shebang>     ← reads /tmp/aml-setup/slm_conda_sft.yaml
#     conda env create -f /tmp/aml-setup/slm_conda_sft.yaml
#     conda activate $ENV_NAME
#     conda install ipykernel
#     sudo -u azureuser -i ENV_NAME="$ENV_NAME" bash << 'EOF'
#       python -m ipykernel install --user --name $ENV_NAME
#     EOF
combined_script = "\n".join([
    "#!/bin/bash",
    "set -e",
    "",
    "# Write conda env yaml inline (embedded at CI creation time, no file share needed)",
    "mkdir -p /tmp/aml-setup",
    "cat > /tmp/aml-setup/slm_conda_sft.yaml << 'YAML_HEREDOC'",
    yaml_content.rstrip(),
    "YAML_HEREDOC",
    "",
    setup_body,
])
script_b64 = base64.b64encode(combined_script.encode("utf-8")).decode("utf-8")
print(f"Script : {len(combined_script)} chars → {len(script_b64)} chars base64")

# ── 3. Build CI using the Python SDK entity (all normal params stay here) ─────
surfix  = datetime.datetime.now().strftime("%Y%m%d%H%M")
ci_name = f"nb-t4{surfix}"
ci_size = "Standard_NC4as_T4_v3"
print(f"CI name : {ci_name}  (length: {len(ci_name)})")

ci = ComputeInstance(
    name=ci_name,
    size=ci_size,
    idle_time_before_shutdown="PT15M",
    idle_time_before_shutdown_minutes=15,
    enable_node_public_ip=True,
    enable_os_patching=False,
    enable_root_access=True,
    enable_sso=True,
    release_quota_on_stop=False,
    ssh_public_access_enabled=True,
    ssh_settings=ComputeInstanceSshSettings(
        ssh_key_value=ssh_public_key_content,
    ),
    tags={"app": "finetuning", "domain": "ml"},
)

# ── 4. Serialize the SDK object to an ARM REST body dict ──────────────────────
arm_body = ci._to_rest_object().serialize()
arm_body["location"] = workspace_location

# ── 5. Inject the inline creationScript (not exposed by the SDK entity) ───────
arm_body["properties"]["properties"]["setupScripts"] = {
    "scripts": {
        "creationScript": {
            "scriptSource": "inline",
            "scriptData": script_b64,
            "timeout": "25m",   # format: "<float>m" — NOT ISO 8601 PT25M
        }
    }
}

# ── 6. PUT via ARM REST (same endpoint the SDK uses internally) ───────────────
token = credential.get_token("https://management.azure.com/.default").token
arm_url = (
    f"https://management.azure.com/subscriptions/{settings.subscription_id}"
    f"/resourceGroups/{settings.resource_group}"
    f"/providers/Microsoft.MachineLearningServices"
    f"/workspaces/{settings.workspace}"
    f"/computes/{ci_name}?api-version=2024-04-01"
)

start_time = datetime.datetime.now()
try:
    resp = requests.put(
        arm_url,
        headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
        json=arm_body,
        timeout=30,
    )
    resp.raise_for_status()
    print(f"✅ CI creation accepted (HTTP {resp.status_code}) — polling every 30s ...")

    max_wait_s = 35 * 60
    elapsed = 0
    state = "Unknown"
    while elapsed < max_wait_s:
        _time.sleep(30)
        elapsed += 30
        try:
            ci_state = ml_client.compute.get(ci_name)
            state = ci_state.provisioning_state or "Unknown"
            print(f"  [{elapsed // 60:2d}m] {state}")
            if state in ("Succeeded", "Failed", "Canceled"):
                break
        except Exception:
            print(f"  [{elapsed // 60:2d}m] not yet visible ...")

    icon = "✅" if state == "Succeeded" else "❌"
    print(f"{icon} Final state : {state}")

except requests.HTTPError as e:
    print(f"❌ HTTP {e.response.status_code}: {e.response.text}")
except Exception as e:
    print(f"❌ Error : {e}")

end_time = datetime.datetime.now()
print(f"Elapsed time : {end_time - start_time}")


Script : 1750 chars → 2336 chars base64
CI name : nb-t4202604012109  (length: 17)
✅ CI creation accepted (HTTP 201) — polling every 30s ...
  [ 0m] Creating
  [ 1m] Creating
  [ 1m] Creating
  [ 2m] Creating
  [ 2m] Creating
  [ 3m] Creating
  [ 3m] Creating
  [ 4m] Creating
  [ 4m] Creating
  [ 5m] Creating
  [ 5m] Creating
  [ 6m] Creating
  [ 6m] Creating
  [ 7m] Creating
  [ 7m] Creating
  [ 8m] Creating
  [ 8m] Creating
  [ 9m] Succeeded
✅ Final state : Succeeded
Elapsed time : 0:09:11.690154


In [ ]:
# if the previous cell fails after CI creation, you can still see the CI in the portal and check the setup script logs under "Boot Diagnostics" → "Serial log" for troubleshooting.
# fetch the log via ARM REST API or SDK

In [31]:
iterator = ml_client.compute.list()
last_ci_name = ""
for item in iterator:
    # type amlcompute is compute cluster
    # type computeinstance is compute instance
    if item.type in ["computeinstance"]:
        print(item.name, item.location, item.type)
        last_ci_name = item.name

notebook-t4-sw swedencentral computeinstance
nb-t4202604012109 swedencentral computeinstance


In [32]:
import json
ci_basic_name = last_ci_name
ci_basic_state = ml_client.compute.get(ci_basic_name)

print(f"compute instance '{ci_basic_state.name}' details:")
print(json.dumps(ci_basic_state.os_image_metadata.__dict__, indent=2, default=str))

compute instance 'nb-t4202604012109' details:
{
  "_is_latest_os_image_version": true,
  "_current_image_version": "26.01.05",
  "_latest_image_version": "26.01.05"
}


In [10]:
import json
ci_basic_name = last_ci_name
ci_basic_state = ml_client.compute.get(ci_basic_name)

print(f"compute instance '{ci_basic_state.name}' details:")
print(json.dumps(ci_basic_state.os_image_metadata.__dict__, indent=2, default=str))

compute instance 'nb-t4202603310016' details:
{
  "_is_latest_os_image_version": true,
  "_current_image_version": "26.01.05",
  "_latest_image_version": "26.01.05"
}


In [12]:
# stop
# from azure.core.exceptions import OperationFailed
try:
    ml_client.compute.begin_stop(ci_basic_name).wait()
    print(f"✅ Compute instance '{ci_basic_name}' stopped successfully.")
except Exception as e:
    print(f"❌ Failed to stop compute instance '{ci_basic_name}': {e}")

✅ Compute instance 'nb-t4202603310016' stopped successfully.


In [13]:
try:
    ml_client.compute.begin_delete(ci_basic_name).wait()
    print(f"✅ Compute instance '{ci_basic_name}' deleted successfully.")
except Exception as e:
    print(f"❌ Failed to delete compute instance '{ci_basic_name}': {e}")

✅ Compute instance 'nb-t4202603310016' deleted successfully.
